In [ ]:
# split_match_data.py
import shutil
import glob
import random
import os
import argparse

def get_argparser():
    parser = argparse.ArgumentParser()
    # Datset Options
    parser.add_argument("--sourceImgRoot", type=str, default='AnBan_split/val', help="path to Train Image Dataset")
    parser.add_argument("--trainVal", action='store_true', default=False, help="")
    parser.add_argument("--valTest", action='store_true', default=False, help="") # 默认 False，存在这个 --valTest 时就会变成 True
    
    return parser

def splitData(oriRoot, image_list, x_img_source_dir, y_annot_source_dir, x_img_target_dir, y_annot_target_dir, rate):
    imageNumber = len(image_list)
    PathList = []

    PathList.append([y_annot_source_dir, y_annot_target_dir])

    set_val_number = int(rate * imageNumber)
    print('set_val_number', set_val_number)
    val_list = random.sample(image_list, set_val_number)
    for val in val_list:
        # filename = val.split("/")[-1]
        filename = val.split("\\")[-1]
        train_image_path = os.path.join(x_img_source_dir, filename).replace('\\', '/')
        val_image_path = os.path.join(x_img_target_dir, filename).replace('\\', '/')

        train_annot_path = os.path.join(y_annot_source_dir, filename[:-3] + 'png').replace('\\', '/')
        val_annot_path = os.path.join(y_annot_target_dir, filename[:-3] + 'png').replace('\\', '/')

        for path in PathList[0]:
            path = os.path.join(oriRoot, path).replace('\\', '/')
            if not os.path.isdir(path): # 如果目标不存在，则创建
                os.makedirs(path)

        shutil.move(train_image_path, val_image_path) # 原本所有文件都放在 源文件夹里，取一部分放到 目标文件夹 里
        shutil.move(train_annot_path, val_annot_path)



def main():
    opts = get_argparser().parse_args()
    oriRoot = os.getcwd()
    path = os.path.join(oriRoot, opts.sourceImgRoot).replace('\\', '/') # path 是 源文件图片的文件。无关 其他文件如 xml 或 mask 或 其他的存放位置
    glob.escape(path)
    image_list = glob.glob(os.path.join(glob.escape(path), '*.jpg'))
    print('image_list', image_list)

    x_train_dir = "AnBan_split/train"
    y_train_dir = "AnBan_split/trainannot"

    x_valid_dir = "AnBan_split/val"
    y_valid_dir = "AnBan_split/valannot"

    x_test_dir = 'AnBan_split/test'
    y_test_dir = 'AnBan_split/testannot'

    if opts.trainVal == True:
        # 训练集和验证集
        splitData(oriRoot, image_list, x_train_dir, y_train_dir, x_valid_dir, y_valid_dir, 0.2)

    if opts.valTest == True:
        # 验证集和测试集
        splitData(oriRoot, image_list, x_valid_dir, y_valid_dir, x_test_dir, y_test_dir, 0.5)


if __name__ == '__main__':
    main()


# 从 train 目录中随机抽取 20% 的数据，放到 val 目录中
# python splitData.py --sourceImgRoot 'AnBan_split/train' --trainVal


# 从 val 目录中随机抽取 50% 的数据，放到 test 目录中
# python splitData.py --sourceImgRoot 'AnBan_split/val' --valTest